In [1]:
# --- 1. Imports ---
import pandas as pd
import numpy as np
from rdkit import Chem
from rdkit.Chem import Descriptors, Fragments, SaltRemover
from mordred import Calculator, descriptors
from sklearn.metrics import r2_score, mean_squared_error, mean_absolute_error
from xgboost import XGBRegressor
import feyn
from tqdm import tqdm
import logging
from func_timeout import func_timeout, FunctionTimedOut
from sklearn.model_selection import GridSearchCV, KFold
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import RandomForestRegressor, VotingRegressor
import lightgbm as lgb
print('✅ All libraries imported.')

This version of Feyn and the QLattice is available for academic, personal, and non-commercial use. By using the community version of this software you agree to the terms and conditions which can be found at https://abzu.ai/eula.

✅ All libraries imported.


In [2]:
# --- 2. Load OP2 Dataset ---
train_file = './dw_data/Opt2_basic_tr.csv'
test_file = './dw_data/Opt2_basic_tst.csv'
target = 'pKa'
smiles_col = 'OriginalSmiles'

print('Loading datasets...')
train_df = pd.read_csv(train_file)
test_df = pd.read_csv(test_file)
for df in [train_df, test_df]:
    df[target] = pd.to_numeric(df[target], errors='coerce')
print('✅ Datasets loaded.')

Loading datasets...
✅ Datasets loaded.


In [3]:
# --- 3. Salt Removal + SMILES to Mol ---
print('Performing salt removal and converting SMILES to molecule objects...')
saltRemover = SaltRemover.SaltRemover(defnFilename='./Salts.txt')
for df in [train_df, test_df]:
    df['Mol'] = df[smiles_col].astype(str).apply(lambda s: saltRemover.StripMol(Chem.MolFromSmiles(s)))
print('✅ Salt removal complete.')

Performing salt removal and converting SMILES to molecule objects...
✅ Salt removal complete.


In [4]:
# --- 4. Descriptor Computation Functions ---
print('Defining descriptor computation functions...')
calc = Calculator(descriptors, ignore_3D=True)

def safe_call(func, mol, timeout=1):
    try:
        return func_timeout(timeout, func, args=(mol,))
    except (FunctionTimedOut, Exception):
        return np.nan

def compute_rdkit_descriptors(mol):
    descriptor_funcs = {name: func for name, func in Descriptors.descList}
    if mol is None or Chem.MolToSmiles(mol) == '':
        return None
    return {name: safe_call(func, mol, timeout=1) for name, func in descriptor_funcs.items()}

def compute_fragment_counts(mol):
    frag_funcs = {name: func for name, func in Fragments.__dict__.items() if callable(func) and name.startswith('fr_')}
    if mol is None or Chem.MolToSmiles(mol) == '':
        return None
    return {name: safe_call(func, mol, timeout=1) for name, func in frag_funcs.items()}

def compute_descriptors_for_df(df):
    print('Computing descriptors for DataFrame...')
    # Filter out rows with invalid molecules
    valid = df['Mol'].apply(lambda mol: (mol is not None) and (Chem.MolToSmiles(mol) != ''))
    df_valid = df[valid].reset_index(drop=True)
    mols = df_valid['Mol'].tolist()
    
    rdkit_list, frag_list = [], []
    for mol in tqdm(mols, total=len(mols), desc='Computing Descriptors'):
        rdkit_desc = compute_rdkit_descriptors(mol)
        frag_desc = compute_fragment_counts(mol)
        rdkit_list.append(rdkit_desc if rdkit_desc is not None else {})
        frag_list.append(frag_desc if frag_desc is not None else {})
    rdkit_df = pd.DataFrame(rdkit_list)
    frag_df = pd.DataFrame(frag_list)
    
    # Compute Mordred descriptors on valid molecules only
    mordred_df = calc.pandas(mols)
    
    full_desc_df = pd.concat([rdkit_df, frag_df, mordred_df], axis=1)
    non_zero_std_cols = full_desc_df.std(numeric_only=True)
    full_desc_df = full_desc_df[non_zero_std_cols[non_zero_std_cols > 0].index]
    full_desc_df = full_desc_df.apply(pd.to_numeric, errors='coerce')
    
    full_df = pd.concat([df_valid[[target]].reset_index(drop=True), 
                          full_desc_df.reset_index(drop=True)], axis=1)
    return full_df.dropna()

print('✅ Descriptor functions defined.')

Defining descriptor computation functions...
✅ Descriptor functions defined.


In [5]:
# --- 5. Generate Descriptors ---
print('Generating descriptors for training and test datasets...')
train_data = compute_descriptors_for_df(train_df)
test_data = compute_descriptors_for_df(test_df)
print('✅ Descriptor generation complete.')

Generating descriptors for training and test datasets...
Computing descriptors for DataFrame...


 25%|███████████████████▍                                                           | 623/2527 [00:07<00:20, 91.00it/s]

C:\Users\Fahad\anaconda3\envs\mainresearch\lib\site-packages\numpy\core\fromnumeric.py:86: RuntimeWarning: overflow encountered in reduce
  return ufunc.reduce(obj, axis, dtype, out, **passkwargs)


 96%|██████████████████████████████████████████████████████████████████████████▎  | 2438/2527 [00:35<00:00, 111.03it/s]

C:\Users\Fahad\anaconda3\envs\mainresearch\lib\site-packages\numpy\core\fromnumeric.py:86: RuntimeWarning: overflow encountered in reduce
  return ufunc.reduce(obj, axis, dtype, out, **passkwargs)


...

Computing descriptors for DataFrame...


 56%|████████████████████████████████████████████▋                                   | 471/843 [00:07<00:09, 40.10it/s]

C:\Users\Fahad\anaconda3\envs\mainresearch\lib\site-packages\numpy\core\fromnumeric.py:86: RuntimeWarning: overflow encountered in reduce
  return ufunc.reduce(obj, axis, dtype, out, **passkwargs)


100%|████████████████████████████████████████████████████████████████████████████████| 843/843 [00:12<00:00, 70.07it/s]


✅ Descriptor generation complete.


In [6]:
# --- 6. Sanitize and Clean Column Names ---
print('Sanitizing descriptor datasets...')

def sanitize(df):
    df = df.loc[:, df.columns.notna()]      # Drop unnamed/NaN columns
    df = df.loc[:, df.notna().all()]          # Drop columns with any NaNs
    df.columns = [str(col) for col in df.columns]  # Convert all to strings
    df = df.loc[:, ~df.columns.duplicated()]  # Remove duplicate columns
    return df.reset_index(drop=True)

train_data = sanitize(train_data)
test_data = sanitize(test_data)
print('✅ Descriptor datasets sanitized.')

Sanitizing descriptor datasets...
✅ Descriptor datasets sanitized.


In [7]:
# --- 7. Feature Alignment and Scaling ---
print('Aligning features between training and test datasets...')

X_train = train_data.drop(columns=[target])
X_test = test_data.drop(columns=[target])

# Add missing columns in test set
missing_cols = set(X_train.columns) - set(X_test.columns)
for col in missing_cols:
    X_test[col] = 0

# Remove extra columns from test set
extra_cols = set(X_test.columns) - set(X_train.columns)
if extra_cols:
    X_test = X_test.drop(columns=list(extra_cols))

# Ensure same column order
X_test = X_test[X_train.columns]

# Apply scaling
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

y_train = train_data[target]
y_test = test_data[target]

print('X_train shape:', X_train.shape, 'X_test shape:', X_test.shape)
print('✅ Feature alignment and scaling complete.')

Aligning features between training and test datasets...
X_train shape: (2518, 1137) X_test shape: (841, 1137)
✅ Feature alignment and scaling complete.


In [8]:
# --- 8. Hyperparameter Tuning for XGBoost ---
print('Starting hyperparameter tuning with GridSearchCV for XGBoost...')

param_grid = {
    'n_estimators': [300],
    'max_depth': [3],
    'learning_rate': [0.1],
    'subsample': [0.8],
    'colsample_bytree': [0.8],
    'reg_lambda': [1]
}

xgb = XGBRegressor(objective='reg:squarederror', random_state=42)
grid_search = GridSearchCV(xgb, param_grid, cv=5, scoring='r2', n_jobs=5, verbose=2)
grid_search.fit(X_train_scaled, y_train)

print('✅ Hyperparameter tuning complete.')
print('Best parameters:', grid_search.best_params_)
print('Best CV R²:', grid_search.best_score_)

best_xgb = grid_search.best_estimator_

Starting hyperparameter tuning with GridSearchCV for XGBoost...
Fitting 5 folds for each of 1 candidates, totalling 5 fits
✅ Hyperparameter tuning complete.
Best parameters: {'colsample_bytree': 0.8, 'learning_rate': 0.1, 'max_depth': 3, 'n_estimators': 300, 'reg_lambda': 1, 'subsample': 0.8}
Best CV R²: -6.037084923017441


In [9]:
# --- 9. Train Best XGBoost with Early Stopping and Evaluate ---
print('Training best XGBoost model with early stopping...')

best_xgb.fit(
    X_train_scaled, y_train,
    eval_set=[(X_test_scaled, y_test)],
    eval_metric='rmse',
    early_stopping_rounds=50,
    verbose=True
)
print('✅ XGBoost training with early stopping complete.')

train_preds = best_xgb.predict(X_train_scaled)
test_preds = best_xgb.predict(X_test_scaled)

train_r2 = r2_score(y_train, train_preds)
train_mse = mean_squared_error(y_train, train_preds)
train_mae = mean_absolute_error(y_train, train_preds)
test_r2 = r2_score(y_test, test_preds)
test_mse = mean_squared_error(y_test, test_preds)
test_mae = mean_absolute_error(y_test, test_preds)

print('Training Metrics: R2: {:.4f}, MSE: {:.4f}, MAE: {:.4f}'.format(train_r2, train_mse, train_mae))
print('Test Metrics: R2: {:.4f}, MSE: {:.4f}, MAE: {:.4f}'.format(test_r2, test_mse, test_mae))

Training best XGBoost model with early stopping...
[0]	validation_0-rmse:6.14764
[1]	validation_0-rmse:5.62614
[2]	validation_0-rmse:5.16214
[3]	validation_0-rmse:4.74793
[4]	validation_0-rmse:4.38519
[5]	validation_0-rmse:4.06680
[6]	validation_0-rmse:3.78582
[7]	validation_0-rmse:3.53764
[8]	validation_0-rmse:3.31730
[9]	validation_0-rmse:3.12589
[10]	validation_0-rmse:2.95545
[11]	validation_0-rmse:2.80982
[12]	validation_0-rmse:2.68892
[13]	validation_0-rmse:2.58641
[14]	validation_0-rmse:2.49932
[15]	validation_0-rmse:2.42418
[16]	validation_0-rmse:2.34879
[17]	validation_0-rmse:2.28766
[18]	validation_0-rmse:2.23295


C:\Users\Fahad\anaconda3\envs\mainresearch\lib\site-packages\xgboost\sklearn.py:835: UserWarning: `eval_metric` in `fit` method is deprecated for better compatibility with scikit-learn, use `eval_metric` in constructor or`set_params` instead.
  warnings.warn(
C:\Users\Fahad\anaconda3\envs\mainresearch\lib\site-packages\xgboost\sklearn.py:835: UserWarning: `early_stopping_rounds` in `fit` method is deprecated for better compatibility with scikit-learn, use `early_stopping_rounds` in constructor or`set_params` instead.
  warnings.warn(


[19]	validation_0-rmse:2.19233
[20]	validation_0-rmse:2.15639
[21]	validation_0-rmse:2.11137
[22]	validation_0-rmse:2.07746
[23]	validation_0-rmse:2.04435
[24]	validation_0-rmse:2.01564
[25]	validation_0-rmse:1.98797
[26]	validation_0-rmse:1.97009
[27]	validation_0-rmse:1.95792
[28]	validation_0-rmse:1.94065
[29]	validation_0-rmse:1.92951
[30]	validation_0-rmse:1.91884
[31]	validation_0-rmse:1.90553
[32]	validation_0-rmse:1.89344
[33]	validation_0-rmse:1.88113
[34]	validation_0-rmse:1.87469
[35]	validation_0-rmse:1.86145
[36]	validation_0-rmse:1.85205
[37]	validation_0-rmse:1.84278
[38]	validation_0-rmse:1.83312
[39]	validation_0-rmse:1.83063
[40]	validation_0-rmse:1.81927
[41]	validation_0-rmse:1.80942
[42]	validation_0-rmse:1.79697
[43]	validation_0-rmse:1.79170
[44]	validation_0-rmse:1.78166
[45]	validation_0-rmse:1.77359
[46]	validation_0-rmse:1.76648
[47]	validation_0-rmse:1.76018
[48]	validation_0-rmse:1.75156
[49]	validation_0-rmse:1.74753
[50]	validation_0-rmse:1.74369
[51]	val

In [10]:
# --- 10. Compute Feature Importances from XGBoost ---
print('Computing feature importances from best XGBoost model...')
importances = pd.Series(best_xgb.feature_importances_, index=X_train.columns).sort_values(ascending=False)
print('Top 10 Features:\n', importances.head(10))

# Select top N features for QLattice (for example, top 150)
N = 150
top_features = importances.head(N).index.tolist()
print(f'Selected top {N} features for QLattice.')

Computing feature importances from best XGBoost model...
Top 10 Features:
 NssCH2                    0.186408
nBase                     0.076435
AETA_beta                 0.066602
FractionCSP3              0.037571
AETA_dBeta                0.021775
nAcid                     0.017594
GATS1m                    0.011999
NumAromaticCarbocycles    0.011548
fr_aniline                0.011475
BCUTZ-1h                  0.011438
dtype: float32
Selected top 150 features for QLattice.


In [11]:
# --- 11. QLattice Nested CV Grid Search for Best Formula ---
print('Starting nested cross-validation grid search for QLattice...')

# Prepare selected feature datasets for QLattice using top features
train_selected = train_data[[target] + top_features].copy()
test_selected = test_data[[target] + top_features].copy()

ql = feyn.QLattice()

from itertools import product
import tqdm

# Define an extensive parameter grid
param_grid = {
    'n_epochs': [200],
    'max_complexity': [300],
    'criterion': ['aic']
}
param_combinations = list(product(param_grid['n_epochs'], param_grid['max_complexity'], param_grid['criterion']))
print(f"Evaluating {len(param_combinations)} parameter combinations using 5-fold CV...")

kf = KFold(n_splits=5, shuffle=True, random_state=42)
nested_results = []
for epochs, complexity, crit in tqdm.tqdm(param_combinations, desc="QLattice Nested CV"):
    cv_scores = []
    for train_idx, val_idx in kf.split(train_selected):
        train_fold = train_selected.iloc[train_idx]
        val_fold = train_selected.iloc[val_idx]
        
        models = ql.auto_run(train_fold, 
                             output_name=target, 
                             n_epochs=epochs,
                             threads=16, 
                             max_complexity=complexity, 
                             criterion=crit)
        if models:
            try:
                preds = models[0].predict(val_fold)
                score = r2_score(val_fold[target], preds)
                cv_scores.append(score)
            except Exception as e:
                cv_scores.append(np.nan)
        else:
            cv_scores.append(np.nan)
    avg_cv = np.nanmean(cv_scores)
    nested_results.append({ 'n_epochs': epochs, 'max_complexity': complexity, 'criterion': crit, 'cv_r2': avg_cv })

nested_df = pd.DataFrame(nested_results).sort_values(by='cv_r2', ascending=False)
print('Top Nested CV QLattice Results:')
print(nested_df.head())

# Select the best QLattice parameter combination
best_params = nested_df.iloc[0]
print('Best QLattice parameters from nested CV:')
print(best_params)

Starting nested cross-validation grid search for QLattice...


KeyError: "['fr_C_S'] not in index"

In [ ]:
# --- 12. Retrain Best QLattice Model on Full Train Data and Evaluate on Test Data ---
print('Retraining QLattice model using best parameters on full training set...')

best_n_epochs = int(best_params['n_epochs'])
best_max_complexity = int(best_params['max_complexity'])
best_criterion = best_params['criterion']

best_models = ql.auto_run(train_selected, 
                         output_name=target, 
                         n_epochs=best_n_epochs,
                         threads=16, 
                         max_complexity=best_max_complexity, 
                         criterion=best_criterion)

if best_models:
    try:
        final_preds = best_models[0].predict(test_selected)
        final_r2 = r2_score(test_selected[target], final_preds)
        print(f'Final QLattice Model Test R²: {final_r2:.4f}')
    except Exception as e:
        print('Error during final evaluation of QLattice model:', e)
else:
    print('No QLattice model was returned for the best parameters.')

# --- Print QLattice Model Formula and Statistics ---
def print_q_lattice_info(model):
    # Try to extract the formula; adjust method name as required by your QLattice API
    try:
        formula = model.formula()  
    except Exception:
        try:
            formula = model.pretty()
        except Exception:
            formula = "Formula unavailable."
    
    # Try to extract model statistics
    try:
        stats = model.stats() 
    except Exception:
        stats = "Statistics unavailable."
    
    print("\n--- QLattice Model Formula ---")
    print(formula)
    print("\n--- QLattice Model Statistics ---")
    print(stats)

if best_models:
    print_q_lattice_info(best_models[0])


In [ ]:
# --- 13. Export Computed Descriptor Datasets ---
print('Saving computed descriptor datasets to CSV files...')
train_data.to_csv('train_descriptors_op1_basic.csv', index=False)
test_data.to_csv('test_descriptors_op1_basic.csv', index=False)
print('✅ Datasets saved to CSV.')